# 删失数据的极大似然估计

> 对应代码：`MathStatsCode/code_in_notes/Chap.11/MLE_censor.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：对 [1000,10000] 区间删失（左右截断）的对数正态数据分段写出对数似然，用 ml 估计 μ 与 σ²。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.11`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.11")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

`clear` 清空数据（本 notebook 将演示对**区间删失数据**的极大似然估计：数据只报告落在 [1000, 10000] 区间内的取值，超出区间的点只记录了“删失”的事实）。


In [ ]:
%%stata
clear


**定义目标函数**

定义目标似然程序 `ml_censor_obj`：`args lnfj theta1 theta2` 接收每个观测的对数似然与参数（μ、σ²）。核心是对三类观测分段写似然：
- `gen lnx=log10($ML_y1)` 先把原始数据取对数——对数正态模型。对位于区间内部的观测（3<lnx<4），对数似然是普通正态密度 `lnnormalden(...)` 减去 log(√σ²)（对数变换的雅可比项）；
- 恰好等于下端点 1000 的观测（lnx=3），只知道真实值 ≤1000，似然为**左删失**的分布函数值 `lnnormal((3-θ1)/√θ2)`；
- 恰好等于上端点 10000 的（lnx=4），知道真实值 ≥10000，似然为**右删失**的 1-Φ(`log(1-normal(...))`)。
这种“内部用密度、端点用累积概率”的分段写法是删失数据 MLE 的标准套路，也是把自定义似然写得灵活的例子。


In [ ]:
%%stata
cap program drop ml_censor_obj
program ml_censor_obj
	args lnfj theta1 theta2
	tempvar lnx
	quietly{
		gen `lnx'=log10($ML_y1)
		replace `lnfj'=lnnormalden((`lnx'-`theta1')/sqrt(`theta2'))-log(sqrt(`theta2')) if `lnx'>3 & `lnx'<4
		replace `lnfj'=lnnormal((3-`theta1')/sqrt(`theta2')) if `lnx'==3
		replace `lnfj'=log(1-normal((4-`theta1')/sqrt(`theta2'))) if `lnx'==4
	}
end


**进行极大似然估计**

定义并运行外层程序：`program mle_censor, eclass` 用 `syntax varlist(max=1)` 接收数据变量；按 `lf` 模型声明 `ml model lf ml_censor_obj (mu: y=, freeparm) /sigma2` 并搜索、最大化；最后把结果 `ereturn post` 为标准的 e(b)/e(V)。随后 `set seed 8855`、`dgp_mle_censor x, obs(500) mu(3.3) sigma2(1)` 调用 `dgp_mle_censor.ado` 生成500个删失样本（真值 μ=3.3、σ²=1），执行 `mle_censor x` 估计——估计结果应接近真值。


In [ ]:
%%stata
cap program drop mle_censor
program mle_censor, eclass
	syntax varlist(max=1)
	quietly{
		ml model lf ml_censor_obj (mu: `varlist'=, freeparm) /sigma2
		ml search
		noisily: ml maximize
		mat b=e(b)
		mat V=e(V)
		ereturn post b V
	}
end

set seed 8855
dgp_mle_censor x, obs(500) mu(3.3) sigma2(1)
mle_censor x
